# LAB 10 — Tokenization & Span Alignment

Notebook minh họa §§10–13, 15–16. Các phép tính tay nằm trong `calculations.md`. Từ thư mục `lab10`, dùng môi trường cài theo README và chạy từ trên xuống.

In [1]:
import os
import pandas as pd
from IPython.display import display
from qa_model import Example, align_answer, load_tokenizer, make_features, tokenize_pairs

tokenizer = load_tokenizer(os.getenv("LAB10_MODEL_SOURCE") or "google/bert_uncased_L-2_H-128_A-2")
question = "What is the capital of France?"
context = "The capital of France is Paris."

## 10. Tokenization của QA

BERT ghép `[CLS] question [SEP] context [SEP]`. `input_ids` là ID từ vựng; `attention_mask` cho biết token thật (1) và padding (0); `token_type_ids` gán segment 0 cho câu hỏi và 1 cho context theo quy ước của BERT. `sequence_ids()` trả về 0 cho câu hỏi, 1 cho context, `None` cho token đặc biệt. Chỉ đọc `token_type_ids` khi tokenizer thực sự trả về trường này. Ví dụ, DistilBERT và RoBERTa chuẩn không cần trường đó.

In [2]:
pair = tokenizer(question, context, return_offsets_mapping=True)
data = {
    "index": range(len(pair["input_ids"])),
    "token": tokenizer.convert_ids_to_tokens(pair["input_ids"]),
    "input_ids": pair["input_ids"],
    "attention_mask": pair["attention_mask"],
    "sequence_id": [str(part) for part in pair.sequence_ids()],
    "offset_mapping": pair["offset_mapping"],
}
if "token_type_ids" in pair:
    data["token_type_ids"] = pair["token_type_ids"]
display(pd.DataFrame(data))
padded = tokenizer(question, context, padding="max_length", max_length=20)
assert padded["attention_mask"] == [1] * 17 + [0] * 3
print("Attention mask with padding:", padded["attention_mask"])

Attention mask with padding: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0]


,index,token,input_ids,attention_mask,sequence_id,offset_mapping,token_type_ids
0,0,[CLS],101,1,None,"(0, 0)",0
1,1,what,2054,1,0,"(0, 4)",0
2,2,is,2003,1,0,"(5, 7)",0
3,3,the,1996,1,0,"(8, 11)",0
4,4,capital,3007,1,0,"(12, 19)",0
5,5,of,1997,1,0,"(20, 22)",0
6,6,france,2605,1,0,"(23, 29)",0
7,7,?,1029,1,0,"(29, 30)",0
8,8,[SEP],102,1,None,"(0, 0)",0
9,9,the,1996,1,1,"(0, 3)",1


## 11. Offset Mapping

Character index đếm ký tự trong văn bản gốc. Word index đếm từ theo cách tách đã chọn. Token index đếm đơn vị tokenizer trong chuỗi đầu vào, gồm subword và token đặc biệt. Với BERT, offset của cả câu hỏi lẫn context đều tính từ đầu chuỗi tương ứng, nên một khoảng ký tự có thể xuất hiện ở cả hai segment. Chỉ dùng segment context khi gán nhãn.

## 12. Calculation Exercise 3 — Kiểm chứng

Trong context, “Paris” có ký tự đầu ở 25 và ký tự cuối ở 29. Biểu diễn Python là `[25, 30)`. Token context thứ 5 là `paris`, ở vị trí 14 trong cặp BERT.

In [3]:
char_left = context.index("Paris")
char_right = char_left + len("Paris")
paris = Example("paris", question, context, (("Paris", char_left),))
feature = make_features([paris], tokenizer, max_length=64, stride=16)[0]
assert (char_left, char_right) == (25, 30)
assert (feature.start, feature.end) == (14, 14)
assert feature.offsets[feature.start] == (25, 30)
print("Gold characters:", (char_left, char_right), "token position:", feature.start,
      "original text:", context[char_left:char_right])

Gold characters: (25, 30) token position: 14 original text: Paris


## 13. Truncation

Maximum sequence length tính cả câu hỏi và token đặc biệt. Với `truncation="only_second"`, phần bị cắt là context. `stride` là số token context được lặp giữa hai cửa sổ. Chạy ví dụ dưới đây để thấy cửa sổ đầu chỉ giữ phần đầu của đáp án, trong khi cửa sổ kế tiếp giữ được cả cụm.

In [4]:
long_context = "word " * 10 + "New York City " + "word " * 28
long_answer_at = long_context.index("New")
long_example = Example("long", "Where?", long_context, (("New York City", long_answer_at),))
truncated = tokenizer(long_example.question, long_context, max_length=16,
                      truncation="only_second", return_offsets_mapping=True)
truncated_offsets = [
    offset if part == 1 else None
    for part, offset in zip(truncated.sequence_ids(), truncated["offset_mapping"])
]
assert align_answer(truncated_offsets, long_answer_at, long_answer_at + 13) is None

windows = make_features([long_example], tokenizer, max_length=16, stride=4)
def window_range(window):
    offsets = [offset for offset in window.offsets if offset is not None]
    return offsets[0][0], offsets[-1][1]

display(pd.DataFrame([
    {"window": index, "context chars": window_range(window),
     "start token": window.start, "end token": window.end,
     "contains full answer": window.start != window.cls_index}
    for index, window in enumerate(windows)
]))
assert len(set(windows[0].offsets[1:]) & set(windows[1].offsets[1:]) - {None}) == 4

,window,context chars,start token,end token,contains full answer
0,0,"(0, 53)",0,0,False
1,1,"(35, 88)",7,9,True
2,2,"(69, 123)",0,0,False
3,3,"(104, 158)",0,0,False
4,4,"(139, 193)",0,0,False
5,5,"(174, 203)",0,0,False


## 15. Core Implementation — `max_length=384, stride=128`

Gọi fast tokenizer trực tiếp. `input_ids` là ID được gửi vào encoder. `attention_mask` đánh dấu các vị trí encoder được đọc. `offset_mapping` giữ cặp chỉ số ký tự để cắt ra văn bản nguồn. `overflow_to_sample_mapping` cho biết cửa sổ thuộc mẫu gốc nào trong batch. Một mẫu dài có thể tạo nhiều dòng đầu vào nên các ID mẫu có thể lặp.

In [5]:
corpus_context = "word " * 410 + "Paris."
encoded = tokenize_pairs(tokenizer, [question, question], [corpus_context, context],
                         max_length=384, stride=128)
overflow_rows = []
for i, origin in enumerate(encoded["overflow_to_sample_mapping"]):
    offsets = [offset for part, offset in zip(encoded.sequence_ids(i), encoded["offset_mapping"][i])
               if part == 1]
    ids, mask = encoded["input_ids"][i], encoded["attention_mask"][i]
    assert len(ids) == len(mask) == len(encoded["offset_mapping"][i])
    overflow_rows.append({"feature": i, "sample": origin, "tokens": len(ids),
                          "input_ids[:5]": ids[:5], "attention_mask[:5]": mask[:5],
                          "context first offset": offsets[0], "context last offset": offsets[-1]})
assert encoded["overflow_to_sample_mapping"] == [0, 0, 1]
display(pd.DataFrame(overflow_rows))

,feature,sample,tokens,input_ids[:5],attention_mask[:5],context first offset,context last offset
0,0,0,384,"[101, 2054, 2003, 1996, 3007]","[1, 1, 1, 1, 1]","(0, 4)","(1865, 1869)"
1,1,0,176,"[101, 2054, 2003, 1996, 3007]","[1, 1, 1, 1, 1]","(1230, 1234)","(2055, 2056)"
2,2,1,17,"[101, 2054, 2003, 1996, 3007]","[1, 1, 1, 1, 1]","(0, 3)","(30, 31)"


## 16. Answer Span Alignment

Start khớp điều kiện `left ≤ answer_start < right`, còn end khớp `left < answer_end ≤ right`. Hai đầu phải thuộc cùng cửa sổ. Nếu gold bắt đầu hoặc kết thúc giữa token, nhãn giữ token bao phủ ký tự đó. Vì vậy khi giải mã bằng offset của token, span trích có thể dài hơn gold. Nếu span bị cắt hoặc nằm hẳn ngoài cửa sổ, cửa sổ đó mang nhãn `[CLS], [CLS]`. Cửa sổ khác của cùng ví dụ vẫn có thể chứa đáp án.

Kiểm chứng bốn ca trong đề và thêm một ca nhiều token.

In [6]:
checks = [
    ("start inside token", "aris", 26, 30, feature, context, (14, 14), "Paris"),
    ("end inside token", "Pari", 25, 29, feature, context, (14, 14), "Paris"),
    ("truncated", "New York City", long_answer_at, long_answer_at + 13,
     windows[0], long_context, None, ""),
    ("outside window", "New York City", long_answer_at, long_answer_at + 13,
     windows[-1], long_context, None, ""),
    ("whole multi-token span", "New York City", long_answer_at, long_answer_at + 13,
     windows[1], long_context, (7, 9), "New York City"),
]
outcomes = []
for case, gold, left, right, window, source_text, expected, recovered in checks:
    assert source_text[left:right] == gold
    result = align_answer(window.offsets, left, right)
    assert result == expected, (case, result)
    label = result if result is not None else (window.cls_index, window.cls_index)
    reconstruction = "" if result is None else source_text[
        window.offsets[result[0]][0]:window.offsets[result[1]][1]
    ]
    assert reconstruction == recovered
    outcomes.append({"case": case, "Gold": gold, "chars": (left, right),
                     "training target": label, "text recovered": reconstruction})
display(pd.DataFrame(outcomes))

,case,Gold,chars,training target,text recovered
0,start inside token,aris,"(26, 30)","(14, 14)",Paris
1,end inside token,Pari,"(25, 29)","(14, 14)",Paris
2,truncated,New York City,"(50, 63)","(0, 0)",
3,outside window,New York City,"(50, 63)","(0, 0)",
4,whole multi-token span,New York City,"(50, 63)","(7, 9)",New York City
